# Эксперименты на Amazon

Подключите версию данных с каталогом, сплитами и манифестом. Для Kaggle добавьте исходный снимок проекта как вход для ноутбука и задайте пути ниже. Локально можно указать корень репозитория и локальный split. Первый запуск по умолчанию оценивается на validation.

Этот ноутбук запускает одну партию на validation или test; для выбора гиперпараметров и повторов по трём seed используйте scripts/run_study.py по инструкции README.

В таблице результатов HitRate@10 — доля запросов с хотя бы одним релевантным товаром в топ-10.

По умолчанию используются модели и бейзлайны из configs/amazon_train_only.json.

## Результаты готовой серии

Ниже сохранены результаты Amazon test при 100%, 50% и 25% обучающих покупок. Метрики усреднены по трём seed. Таблицы читаются из [test_summary.csv](../docs/results/amazon-train-only/test_summary.csv). Для их просмотра обучение и исходные данные не нужны.

In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

relative_path = Path("docs/results/amazon-train-only/test_summary.csv")
project_roots = (
    Path(os.environ.get("RECSYS_PROJECT_ROOT", "/kaggle/input/complementary-recsys-code")),
    Path.cwd(),
    Path.cwd().parent,
)
summary_file = next(
    (root / relative_path for root in project_roots if (root / relative_path).is_file()),
    None,
)
if summary_file is None:
    raise FileNotFoundError(f"Укажите RECSYS_PROJECT_ROOT с файлом {relative_path}")

summary = pd.read_csv(summary_file)
metrics = {"hit_rate_mean": "HitRate@10", "ndcg_mean": "NDCG@10", "recall_mean": "Recall@10"}
for buy_fraction, selected in summary.groupby("buy_fraction", sort=False):
    display(Markdown(f"**{buy_fraction:.0%} обучающих покупок**"))
    display(
        selected.sort_values(list(metrics), ascending=False)
        .set_index("model")[list(metrics)]
        .rename(columns=metrics)
        .rename_axis("Модель")
        .round(4)
    )

**100% обучающих покупок**

,HitRate@10,NDCG@10,Recall@10
Модель,,,
HOSVD,0.7474,0.5249,0.7248
CP buy-only,0.7061,0.4615,0.6800
CP buy+view,0.6980,0.4505,0.6722
Tucker buy-only,0.6938,0.4444,0.6669
MIG-TF,0.6916,0.4458,0.6650
ComplEx buy+view,0.6916,0.4454,0.6651
DistMult buy-only,0.6910,0.4504,0.6653
ComplEx buy-only,0.6904,0.4495,0.6641
Tucker buy+view,0.6867,0.4393,0.6604


**50% обучающих покупок**

,HitRate@10,NDCG@10,Recall@10
Модель,,,
HOSVD,0.6600,0.4498,0.6350
MIG-TF,0.6280,0.3917,0.6001
DistMult buy+view,0.6233,0.3926,0.5954
Tucker buy+view,0.6200,0.3853,0.5926
DistMult buy-only,0.6199,0.3936,0.5916
Tucker buy-only,0.6172,0.3825,0.5898
TwoTower Mixed,0.6144,0.3789,0.5869
ComplEx buy+view,0.6113,0.3834,0.5850
CP buy+view,0.6090,0.3823,0.5824


**25% обучающих покупок**

,HitRate@10,NDCG@10,Recall@10
Модель,,,
HOSVD,0.6185,0.4044,0.5941
TwoTower Mixed,0.6111,0.3716,0.5858
MIG-TF,0.5427,0.3242,0.5173
Tucker buy+view,0.5316,0.3148,0.5061
Tucker buy-only,0.5246,0.3093,0.4992
DistMult buy+view,0.4702,0.2894,0.4455
DistMult buy-only,0.4609,0.2837,0.4362
ComplEx buy-only,0.4107,0.2435,0.3878
CP buy-only,0.4056,0.2413,0.3827


## Запуск нового эксперимента

Ниже сохранены результаты отдельного запуска на validation и примеры выдачи. Сводка выше использует test и три seed.

In [2]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path(os.environ.get('RECSYS_PROJECT_ROOT', '/kaggle/input/complementary-recsys-code'))
SPLIT_DIR = Path(os.environ.get('RECSYS_SPLIT_DIR', '/kaggle/input/amazon-complementary-split'))
OUTPUT_ROOT = Path(os.environ.get('RECSYS_OUTPUT_ROOT', '/kaggle/working/runs'))
CODE_REVISION = os.environ.get('RECSYS_CODE_REVISION')
CONFIG = os.environ.get('RECSYS_CONFIG', 'configs/amazon_train_only.json')
PARTITION = os.environ.get('RECSYS_PARTITION', 'validation')

assert (PROJECT_ROOT / 'src/complementary_recsys').is_dir(), PROJECT_ROOT
assert (SPLIT_DIR / 'manifest.json').is_file(), SPLIT_DIR
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [3]:
import torch
from complementary_recsys.data.bundle import SplitBundle
from complementary_recsys.experiment import ExperimentRunner

config = json.loads((PROJECT_ROOT / CONFIG).read_text())
config['training']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
if CODE_REVISION:
    config['code_revision'] = CODE_REVISION
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output = OUTPUT_ROOT / run_id
bundle = SplitBundle.load(SPLIT_DIR)
results = ExperimentRunner(bundle, config, PARTITION).run(output)
display(results)

{"model": "Tucker buy-only", "recall@10": 0.6902213457243714, "ndcg@10": 0.4629270696140225, "hit_rate@10": 0.7146747352496218, "queries": 6610, "mean_candidates": 162.16944024205748, "coverage": 0.9215112226404573, "parameters": 641024, "fit_seconds": 8.819194457959384, "evaluation_seconds": 0.8690654579550028}


{"model": "Tucker buy+view", "recall@10": 0.6758237879115337, "ndcg@10": 0.44951586854941833, "hit_rate@10": 0.7013615733736762, "queries": 6610, "mean_candidates": 162.16944024205748, "coverage": 0.9215112226404573, "parameters": 642048, "fit_seconds": 11.01196216698736, "evaluation_seconds": 0.7402194170281291}


{"model": "TwoTower Mixed", "recall@10": 0.6032168071464592, "ndcg@10": 0.376608962574244, "hit_rate@10": 0.6288956127080182, "queries": 6610, "mean_candidates": 162.16944024205748, "coverage": 0.9215112226404573, "parameters": 361760, "fit_seconds": 10.715208999812603, "evaluation_seconds": 5.626972707919776}


{"model": "HOSVD", "recall@10": 0.7350072040919242, "ndcg@10": 0.5347194263130722, "hit_rate@10": 0.7571860816944024, "queries": 6610, "mean_candidates": 162.16944024205748, "coverage": 0.9215112226404573, "parameters": 1282052, "fit_seconds": 0.8115322501398623, "evaluation_seconds": 0.8340343339368701}


{"model": "DistMult buy-only", "recall@10": 0.6793500108061379, "ndcg@10": 0.46694611482046344, "hit_rate@10": 0.705446293494705, "queries": 6610, "mean_candidates": 162.16944024205748, "coverage": 0.9215112226404573, "parameters": 640032, "fit_seconds": 4.108911792282015, "evaluation_seconds": 1.0816596248187125}


{"model": "DistMult buy+view", "recall@10": 0.6770547871190836, "ndcg@10": 0.4520130409051196, "hit_rate@10": 0.7034795763993948, "queries": 6610, "mean_candidates": 162.16944024205748, "coverage": 0.9215112226404573, "parameters": 640064, "fit_seconds": 5.045343458186835, "evaluation_seconds": 1.196636499837041}


... (4 outputs left out)

{"model": "MIG-TF", "recall@10": 0.6780491319069231, "ndcg@10": 0.45274865573043005, "hit_rate@10": 0.7030257186081694, "queries": 6610, "mean_candidates": 162.16944024205748, "coverage": 0.9215112226404573, "parameters": 722056, "fit_seconds": 14.81235958263278, "evaluation_seconds": 1.984128958079964}


{"model": "Graph buy-buy", "recall@10": 0.5117329443123695, "ndcg@10": 0.3667061938047568, "hit_rate@10": 0.5369137670196672, "queries": 6610, "mean_candidates": 162.16944024205748, "coverage": 0.9215112226404573, "parameters": 0, "fit_seconds": 0.007952625397592783, "evaluation_seconds": 1.9509207918308675}


{"model": "Graph mixed paths", "recall@10": 0.5575021612275772, "ndcg@10": 0.40616786745956157, "hit_rate@10": 0.583963691376702, "queries": 6610, "mean_candidates": 162.16944024205748, "coverage": 0.9215112226404573, "parameters": 0, "fit_seconds": 0.010407458059489727, "evaluation_seconds": 3.9584283749572933}


,model,recall@10,ndcg@10,hit_rate@10,queries,mean_candidates,coverage,parameters,fit_seconds,evaluation_seconds
0,HOSVD,0.735007,0.534719,0.757186,6610,162.16944,0.921511,1282052,0.811532,0.834034
1,CP buy-only,0.701446,0.474924,0.726778,6610,162.16944,0.921511,1280032,5.157674,1.438908
2,Tucker buy-only,0.690221,0.462927,0.714675,6610,162.16944,0.921511,641024,8.819194,0.869065
3,ComplEx buy-only,0.681256,0.462443,0.708472,6610,162.16944,0.921511,640032,3.711328,1.175110
4,DistMult buy-only,0.679350,0.466946,0.705446,6610,162.16944,0.921511,640032,4.108912,1.081660
5,ComplEx buy+view,0.678873,0.453977,0.705446,6610,162.16944,0.921511,640064,5.056464,1.276582
6,DistMult buy+view,0.677055,0.452013,0.703480,6610,162.16944,0.921511,640064,5.045343,1.196636
7,MIG-TF,0.678049,0.452749,0.703026,6610,162.16944,0.921511,722056,14.812360,1.984129
8,CP buy+view,0.676357,0.459853,0.702421,6610,162.16944,0.921511,1280064,6.211680,1.284330
9,Tucker buy+view,0.675824,0.449516,0.701362,6610,162.16944,0.921511,642048,11.011962,0.740219


In [4]:
import pandas as pd

recommendations = pd.read_csv(output / 'recommendations.csv.gz', dtype={'source_id': str})
display(recommendations.head(10))
print(output)

,model,source_id,target_category,user_id,top10,relevant,recall@10,ndcg@10,hit_rate@10
0,Tucker buy-only,0000098906,Cabasas,NaN,"[""B0002F51Y8"", ""B0002F51YS"", ""B01DKEDZFO"", ""B0...","[""B0002F51YS""]",1.0,0.630930,1
1,Tucker buy-only,0000098906,Concert Percussion,NaN,"[""B00AZZ1AJK"", ""B000UJE6UG"", ""B000J03ARG""]","[""B00AZZ1AJK""]",1.0,1.000000,1
2,Tucker buy-only,0000098906,Guiros,NaN,"[""B0002IHDYG"", ""B001XW31UK"", ""B00DP1YFTI"", ""B0...","[""B0149IWDS0""]",1.0,0.430677,1
3,Tucker buy-only,0000098906,Mouth Harps,NaN,"[""B015Y18ULU"", ""B0012Q7B5S"", ""B00282WZG0"", ""B0...","[""B015Y18ULU""]",1.0,1.000000,1
4,Tucker buy-only,0000098906,Singing Bowls,NaN,"[""B00B1BD2ZW"", ""B00NMEYUPM"", ""B01A6B0ICC"", ""B0...","[""B01A6B0ICC""]",1.0,0.500000,1
5,Tucker buy-only,0692218645,Guitar & Bass Accessories,NaN,"[""B0002D0CQC"", ""B01F7EAMYC"", ""B005CX4GLE"", ""B0...","[""B0164DKTFM""]",0.0,0.000000,0
6,Tucker buy-only,0825603285,Whistles,NaN,"[""B0002I8Y6W"", ""B017WBDEUC"", ""B000VPHCDM"", ""B0...","[""B0036H5FOA"", ""B00VK4YXF8""]",0.5,0.177239,1
7,Tucker buy-only,1480360295,Picks,NaN,"[""B00HFSHFPI"", ""B001PGXHXA"", ""B005SUVENO"", ""B0...","[""B00HFSHFPI""]",1.0,1.000000,1
8,Tucker buy-only,3959828292,Equalization,NaN,"[""B000KIRT6A"", ""B000RPWNS0"", ""B001XURLDA"", ""B0...","[""B000KIRT6A""]",1.0,1.000000,1
9,Tucker buy-only,3959828292,Powered Mixers,NaN,"[""B000J5Y282"", ""B00JRGHEZ2"", ""B000J5Y214"", ""B0...","[""B000J5UEGQ""]",1.0,0.430677,1


runs/notebook-rerun/20261121T152448257092Z
